# Практика · PEFT і LoRA: скільки ваг насправді треба чіпати

> Теорія — у [lecture.html](lecture.html) · Тест — [quiz.html](quiz.html) ·
> Домашнє завдання — [homework.html](homework.html)

Зошит самодостатній: він пояснює все, що робить, і читати лекцію заздалегідь не треба.

**Задача.** У нас є багато повідомлень програм українською. Частина з них
повідомляє про **збій** («не вдалося відкрити файл»), решта — ні. Треба навчити
модель це розрізняти.

Робити будемо так, як роблять із великими мовними моделями:

1. **передтренуємо** маленький трансформер як мовну модель на всьому корпусі —
   він навчиться самої мови, нічого не знаючи про нашу задачу;
2. **донавчимо** його під класифікацію **трьома різними способами**, витративши
   на кожен **однаковий бюджет**: тільки голова · LoRA · усі ваги;
3. **порівняємо**: скільки ваг кожен спосіб чіпає і що з цього виходить.

Дорогою двічі перевіримо власну реалізацію рівністю чисел — саме там, де
теорія обіцяє точну рівність, а не наближену.

> ⏱ Зошит навчає одну мовну модель і **двадцять вісім** маленьких
> класифікаторів. Заміряно: **279 секунд процесорного часу** (з них 122 —
> передтренування) на чотирьох ядрах без відеокарти. Годинник покаже помітно
> більше, бо машина спільна: наш прогін тривав 11 хвилин.

## 0 · Налаштування

Дві дрібниці, які насправді важливі.

**Кількість потоків фіксуємо до імпорту `numpy` і `torch`.** Інакше бібліотеки
розкидають роботу по ядрах, потоки крутяться в очікуванні одне одного, і це
очікування рахується як робота — процесорний час роздувається в десятки разів.

**Час міряємо `time.process_time()`**, а не годинником: годинник показує, скільки
минуло в кімнаті, а нам треба, скільки з'їв наш процес.

In [ ]:
import os
# ці три рядки мусять стояти ДО імпорту numpy і torch, інакше не подіють
for v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[v] = '1'

import sys, glob, gettext, re, math, random, time, collections
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

torch.set_num_threads(1)
STARTED = time.process_time()

print('python  ', sys.version.split()[0])
print('torch   ', torch.__version__)
print('numpy   ', np.__version__)

## 1 · Дані: справжні повідомлення з твоєї машини

Корпус ми не завантажуємо з мережі — він уже лежить у системі. Кожна програма,
перекладена українською, тримає свої переклади у файлі `.mo` в каталозі
`/usr/share/locale/uk/LC_MESSAGES/`. Формат `.mo` — це словник «англійський
оригінал → український переклад», і читати його вміє стандартний модуль
`gettext`.

Чим цей корпус цінний саме тут: він **паралельний**. Український текст ми
подаємо моделі, а мітку («це повідомлення про збій чи ні») беремо з
**англійського оригіналу**, де розробник сам написав `error`, `failed`,
`cannot`. Тобто мітку писала людина, і вона не виведена з того самого тексту,
який класифікуємо.

⚠️ **Це проксі, а не еталонна розмітка.** Правило по ключових словах ловить
`cannot`, але не ловить `could not` чи `is not able to`. Частина повідомлень про
збій дістане мітку «не збій», і стеля задачі через це нижча за одиницю. Ми це
знаємо й кажемо вголос.

In [ ]:
LOCALE_DIR = '/usr/share/locale/uk/LC_MESSAGES'
paths = sorted(glob.glob(os.path.join(LOCALE_DIR, '*.mo')))

if not paths:
    print('❌ Українських перекладів на цій машині немає.')
    print('   Зошит читає', LOCALE_DIR, '— там порожньо.')
    print('   На Debian/Ubuntu спробуй:  sudo apt install language-pack-uk')
    print('   На Fedora переклади ставляться разом із програмами (langpacks-uk).')
    print('   Без корпусу далі йти нікуди — решта клітинок працювати не буде.')
else:
    print(f'знайдено файлів перекладу: {len(paths)}')
    print('перші пʼять:', [os.path.basename(p) for p in paths[:5]])

Тепер читаємо самі повідомлення. Беремо лише довгі (понад 20 символів у
перекладі) — короткі рядки на кшталт «OK» чи «Так» мови не показують.

In [ ]:
messages = []                       # (англійський оригінал, український переклад)
for path in paths:
    try:
        with open(path, 'rb') as f:
            catalog = gettext.GNUTranslations(f)._catalog
    except Exception:
        continue                    # трапляються биті або нестандартні файли — просто пропускаємо
    for source, translated in catalog.items():
        if not isinstance(source, str) or not isinstance(translated, str):
            continue
        # службовий рядок із метаданими каталогу, не текст
        if 'Project-Id' in translated or len(translated) <= 20:
            continue
        messages.append((source, translated))

print(f'повідомлень із перекладом: {len(messages)}')
print()
for source, translated in messages[:3]:
    print(' англ:', source[:70])
    print(' укр :', translated[:70])
    print()

⚠️ **Твої числа не збігатимуться з нашими, і це нормально.** Корпус складається
з того, що встановлено саме на твоїй машині. У нас це було 273 файли; у тебе
буде інше число, інший словник і інший розмір вибірки. Відтворюється **форма**
результату, а не значення. Друкуй свої числа й порівнюй порядок величин.

## 2 · Токенізація й мітки

**Токенізація** — це поділ тексту на одиниці, з якими працює модель. Ми беремо
найпростіший варіант: слова з українських літер, з апострофом усередині
(«обʼєкт» — одне слово, не два). Усе інше — цифри, латиниця, розділові знаки —
відкидаємо.

**Мітка** — одиниця, якщо в **англійському оригіналі** трапилось слово зі
списку «збійних», і нуль інакше.

In [ ]:
TOKEN_RE = re.compile(r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*")
FAIL_RE  = re.compile(r"\b(error|fail|failed|cannot|unable|invalid|denied|corrupt)\b", re.I)

MAX_LEN = 32                        # довші повідомлення відкидаємо: моделі й так вистачить
PAD, BOS, EOS, UNK = 0, 1, 2, 3     # службові номери: заповнювач, початок, кінець, невідоме слово

samples = []                        # (список слів, мітка)
for source, translated in messages:
    words = TOKEN_RE.findall(translated.lower())
    if 4 <= len(words) <= MAX_LEN - 2:      # -2, бо додамо <bos> і <eos>
        samples.append((words, 1 if FAIL_RE.search(source) else 0))

# перемішуємо з фіксованим зерном: у корпусі повідомлення йдуть за програмами,
# і без перемішування в перевірну частину потрапили б лише останні програми
random.Random(0).shuffle(samples)

n_total = len(samples)
share_fail = sum(label for _, label in samples) / n_total
print(f'придатних повідомлень: {n_total}')
print(f'частка «збій»: {share_fail:.4f}   (тобто класи дуже нерівні)')
print()
for words, label in samples[:4]:
    print(('ЗБІЙ    ' if label else 'не збій '), ' '.join(words[:9]))

Ділимо на три частини й будуємо словник.

Три частини, а не дві, — і це принципово. **Навчальна** (80 %) — на ній
вчиться модель. **Відкладена** (10 %) — на ній ми підбиратимемо швидкість
навчання. **Перевірна** (10 %) — до неї не торкаємось, поки все не вирішено;
інакше ми підглянемо у відповідь, і число перестане щось означати.

Словник будуємо **лише з навчальної частини**: інакше модель дізналася б про
слова з перевірної ще до навчання. Слова, що трапились менш ніж пʼять разів,
замінюємо на `<unk>` — на рідкісному слові однаково нічого не вивчиш.

In [ ]:
a, b = int(0.8 * n_total), int(0.9 * n_total)
train_raw, holdout_raw, test_raw = samples[:a], samples[a:b], samples[b:]

counts = collections.Counter(w for words, _ in train_raw for w in words)
itos = ['<pad>', '<bos>', '<eos>', '<unk>'] + [w for w, c in counts.most_common() if c >= 5]
stoi = {w: i for i, w in enumerate(itos)}
VOCAB = len(itos)

def encode(words):
    """Слова -> номери, з рамкою <bos> ... <eos>."""
    return [BOS] + [stoi.get(w, UNK) for w in words] + [EOS]

def prepare(raw):
    return [encode(words) for words, _ in raw], [label for _, label in raw]

train_x, train_y = prepare(train_raw)
hold_x,  hold_y  = prepare(holdout_raw)
test_x,  test_y  = prepare(test_raw)

print(f'словник: {VOCAB} слів')
print(f'навчальна {len(train_x)} · відкладена {len(hold_x)} · перевірна {len(test_x)}')
print()
print('приклад кодування:')
print(' ', ' '.join(train_raw[0][0][:8]))
print(' ', encode(train_raw[0][0])[:10], '...')

Остання дрібниця перед моделлю: у пачці речення різної довжини, а тензор мусить
бути прямокутним. Тому коротші добиваємо номером `PAD`, і модель навчиться його
ігнорувати.

In [ ]:
def pad_batch(sequences):
    """Список списків різної довжини -> прямокутний тензор, добитий PAD."""
    longest = max(len(s) for s in sequences)
    return torch.tensor([s + [PAD] * (longest - len(s)) for s in sequences])

example = pad_batch([train_x[0], train_x[1], train_x[2]])
print('форма пачки:', tuple(example.shape))
print(example[:, :12])

## 3 · Модель: найменший трансформер, який ще працює

Пишемо його руками, а не беремо готовий, бо далі нам треба буде дістатися до
конкретних матриць усередині — а в бібліотечному шарі вони склеєні в одну й
дістати їх поодинці незручно.

Що всередині одного блока:

| матриця | що робить |
|---|---|
| `Wq` | робить **запит**: «що цей токен шукає» |
| `Wk` | робить **ключ**: «що цей токен пропонує» |
| `Wv` | робить **значення**: «що він віддасть, якщо його оберуть» |
| `Wo` | змішує результати голів назад в один вектор |
| `f1`, `f2` | мережа прямого поширення після уваги |

**Увага** порівнює кожен запит із кожним ключем, перетворює схожості на ваги
через `softmax` і бере зважену суму значень. **Маска** робить дві речі:
забороняє дивитись уперед (модель передбачає наступне слово, і підглядати не
можна) і забороняє дивитись на заповнювач `PAD`.

In [ ]:
D_MODEL, N_HEADS = 64, 2

class Block(nn.Module):
    """Один блок трансформера: увага + мережа прямого поширення."""
    def __init__(self, d, heads):
        super().__init__()
        self.heads = heads
        self.d_head = d // heads
        self.Wq = nn.Linear(d, d)
        self.Wk = nn.Linear(d, d)
        self.Wv = nn.Linear(d, d)
        self.Wo = nn.Linear(d, d)
        self.f1 = nn.Linear(d, 4 * d)
        self.f2 = nn.Linear(4 * d, d)
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)

    def attention(self, x, mask):
        batch, length, d = x.shape
        def split_heads(t):
            return t.view(batch, length, self.heads, self.d_head).transpose(1, 2)
        q, k, v = split_heads(self.Wq(x)), split_heads(self.Wk(x)), split_heads(self.Wv(x))
        # схожість запиту з ключем; ділення на корінь стримує розкид перед softmax
        scores = q @ k.transpose(-1, -2) / math.sqrt(self.d_head) + mask
        weights = scores.softmax(-1)
        mixed = (weights @ v).transpose(1, 2).reshape(batch, length, d)
        return self.Wo(mixed)

    def forward(self, x, mask):
        x = x + self.attention(self.norm1(x), mask)     # залишковий звʼязок
        return x + self.f2(F.gelu(self.f1(self.norm2(x))))


class MiniLM(nn.Module):
    """Мовна модель: передбачає наступне слово."""
    def __init__(self, vocab, d=D_MODEL, heads=N_HEADS):
        super().__init__()
        self.emb = nn.Embedding(vocab, d, padding_idx=PAD)
        self.pos = nn.Embedding(MAX_LEN, d)
        self.block = Block(d, heads)
        self.norm_final = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, vocab)

    def hidden(self, x):
        """Вектори токенів після блока — це «тіло» моделі."""
        length = x.size(1)
        # верхній трикутник у мінус нескінченність: не дивитись уперед
        causal = torch.triu(torch.full((length, length), float('-inf')), 1)
        # плюс заборона дивитись на заповнювач
        pad_mask = torch.where(x == PAD, float('-inf'), 0.0)[:, None, None, :]
        mask = causal.unsqueeze(0).unsqueeze(0) + pad_mask
        return self.norm_final(self.block(self.emb(x) + self.pos(torch.arange(length)), mask))

    def forward(self, x):
        return self.lm_head(self.hidden(x))


torch.manual_seed(0)
probe_model = MiniLM(VOCAB)
total = sum(p.numel() for p in probe_model.parameters())
body_only = total - probe_model.lm_head.weight.numel() - probe_model.lm_head.bias.numel()
print(f'усього ваг у мовній моделі: {total}')
print(f'з них у вихідній проєкції:  {total - body_only}  (класифікації вона не потрібна)')

## 4 · Передтренування: модель вчить мову, а не нашу задачу

Це найдовша клітинка зошита. Модель дивиться на речення й на кожній позиції
намагається вгадати **наступне слово**. Про «збій / не збій» вона не знає нічого.

Якість мовної моделі міряємо в **натах** — це середня кількість «здивування» на
одне слово (мінус логарифм імовірності правильного слова, у натуральних
логарифмах). Менше — краще.

Щоб число щось означало, поруч мусить стояти **рубіж**: скільки нат дасть
модель, яка не дивиться на контекст узагалі, а просто вгадує за частотою слів
у корпусі. Це **уніграмний рубіж**, і він рахується без жодного навчання.

In [ ]:
PRETRAIN_STEPS = 600
BATCH = 64

torch.manual_seed(0)
base_lm = MiniLM(VOCAB)
optimizer = torch.optim.AdamW(base_lm.parameters(), lr=3e-3, weight_decay=0.01)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)
sampler = random.Random(0)
indices = list(range(len(train_x)))

t0 = time.process_time()
for step in range(PRETRAIN_STEPS):
    batch = pad_batch([train_x[j] for j in sampler.sample(indices, BATCH)])
    # вхід — усе, крім останнього токена; ціль — усе, крім першого
    logits = base_lm(batch[:, :-1])
    loss = loss_fn(logits.reshape(-1, VOCAB), batch[:, 1:].reshape(-1))
    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(base_lm.parameters(), 1.0)
    optimizer.step()
    if (step + 1) % 200 == 0:
        print(f'  крок {step+1}: втрата {loss.item():.4f} · '
              f'{time.process_time()-t0:.0f} с процесорних', flush=True)

print(f'передтренування скінчено за {time.process_time()-t0:.0f} с процесорного часу')

In [ ]:
@torch.no_grad()
def language_nats(model, sequences, batch=128):
    """Середнє «здивування» моделі на одне слово, у натах."""
    model.eval()
    summed = nn.CrossEntropyLoss(ignore_index=PAD, reduction='sum')
    total_loss = total_tokens = 0
    for i in range(0, len(sequences), batch):
        x = pad_batch(sequences[i:i + batch])
        total_loss += summed(model(x[:, :-1]).reshape(-1, VOCAB), x[:, 1:].reshape(-1)).item()
        total_tokens += (x[:, 1:] != PAD).sum().item()
    model.train()
    return total_loss / total_tokens

# рубіж: модель без контексту, яка знає лише частоти слів навчальної частини
token_counts = collections.Counter(t for seq in train_x for t in seq[1:])
total_count = sum(token_counts.values())
unigram_nats = 0.0
seen = 0
for seq in test_x:
    for t in seq[1:]:
        p = token_counts.get(t, 0.5) / total_count      # 0.5 замість нуля для небачених
        unigram_nats += -math.log(p)
        seen += 1
unigram_nats /= seen

model_nats = language_nats(base_lm, test_x)
print(f'уніграмний рубіж (без контексту): {unigram_nats:.4f} ната')
print(f'наша модель:                      {model_nats:.4f} ната')
print(f'запас: {unigram_nats - model_nats:.4f} ната — модель справді читає контекст')

Якщо запас додатний і помітний — модель існує, і донавчати є що. Це не
формальність: гілка, у якої власна стадія підготовки не доведена до кінця,
робить будь-яке подальше порівняння порівнянням із шумом.

## 5 · Три способи донавчити — і LoRA у двадцяти рядках

Тепер головне. Основа готова; треба навчити її нової справи. Спробуємо три
способи, від найощаднішого до найдорожчого.

**1. Тільки голова («лінійна проба»).** Тіло заморожуємо повністю, зверху
ставимо один лінійний шар `64 → 2`. Вчиться рівно `64·2 + 2 = 130` ваг. Це
**рубіж знизу**: якщо якийсь хитрий метод не обганяє цього, він не потрібен.

**2. LoRA.** Тіло теж заморожене, але до матриць `Wq` і `Wv` дописуємо
паралельну гілку: вхід стискається до `r` чисел матрицею `A`, потім
розтискається назад матрицею `B`. Результат додається до виходу замороженого
шару.

**3. Усі ваги.** Звичайне повне донавчання. Це **рубіж зверху**.

Ось уся LoRA. Зверни увагу на два рядки в конструкторі: `A` дістає випадкові
малі числа, `B` — **рівно нулі**. Тоді добуток `B·A` на старті нульовий, і
модель точнісінько така, якою вийшла з передтренування. А занулити обидві не
можна: похідна по `A` містить множник `B`, а по `B` — множник `A`, тож обидві
нульові матриці лишились би нулями назавжди.

In [ ]:
class LoRALinear(nn.Module):
    """Заморожений лінійний шар плюс поправка малого рангу: W·x + (B·A·x)·alpha/r."""

    def __init__(self, base_layer, rank, alpha=None):
        super().__init__()
        self.base = base_layer
        for p in self.base.parameters():
            p.requires_grad = False               # оригінал не вчиться взагалі
        self.rank = rank
        self.scale = (rank if alpha is None else alpha) / rank
        # A — випадкова, B — нулі. Несиметрія тут обовʼязкова, див. текст вище
        self.A = nn.Parameter(torch.randn(rank, base_layer.in_features) * 0.02)
        self.B = nn.Parameter(torch.zeros(base_layer.out_features, rank))

    def forward(self, x):
        # дужки саме такі: спершу стискаємо вхід до rank чисел, і лише потім
        # розтискаємо. Якби ми спершу перемножили B·A, економії не було б
        correction = (x @ self.A.T) @ self.B.T
        return self.base(x) + correction * self.scale

    def merged_weight(self):
        """Одна матриця, що робить те саме, що обидві дороги разом."""
        return self.base.weight + (self.B @ self.A) * self.scale


class Classifier(nn.Module):
    """Тіло мовної моделі + лінійна голова на два класи."""
    def __init__(self, body, d=D_MODEL):
        super().__init__()
        self.body = body
        self.head = nn.Linear(d, 2)

    def forward(self, x):
        vectors = self.body.hidden(x)
        # усереднюємо вектори всіх справжніх токенів, заповнювач не рахуємо
        keep = (x != PAD).float().unsqueeze(-1)
        pooled = (vectors * keep).sum(1) / keep.sum(1).clamp(min=1)
        return self.head(pooled)


BASE_STATE = {k: v.clone() for k, v in base_lm.state_dict().items()}

def build_arm(kind, seed, rank=4):
    """Збирає класифікатор на тій самій основі одним із трьох способів."""
    torch.manual_seed(seed)
    body = MiniLM(VOCAB)
    body.load_state_dict(BASE_STATE)
    del body.lm_head                    # у класифікації вона не бере участі —
                                        # і рахувати її ваги як «навчені» не можна
    if kind != 'full':
        for p in body.parameters():
            p.requires_grad = False
    if kind == 'lora':
        body.block.Wq = LoRALinear(body.block.Wq, rank)
        body.block.Wv = LoRALinear(body.block.Wv, rank)
    return Classifier(body)


def trainable_weights(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for kind in ('probe', 'lora', 'full'):
    m = build_arm(kind, 0)
    print(f'{kind:6} — навчуваних ваг: {trainable_weights(m):8}')

### Перевірка перша: LoRA на старті нічого не міняє

Теорія обіцяє не «майже те саме», а **точну рівність**: поки `B` нульова,
поправка дорівнює нулю, і модель із LoRA видає рівно те, що видавала основа.
Перевіримо це числом на справжніх реченнях.

In [ ]:
plain = build_arm('probe', 0)
with_lora = build_arm('lora', 0)
with_lora.head.load_state_dict(plain.head.state_dict())   # голови мають бути однакові

check_batch = pad_batch(test_x[:64])
with torch.no_grad():
    plain.eval(); with_lora.eval()
    difference = (plain(check_batch) - with_lora(check_batch)).abs().max().item()
    plain.train(); with_lora.train()

print(f'найбільша різниця виходів: {difference:.3e}')
assert difference < 1e-5, 'LoRA з нульовою B мусить бути тотожною основі!'
print('✅ збігається: поправка на старті дорівнює нулю')

## 6 · Скільки це коштує в памʼяті

Тепер порахуємо те, заради чого весь метод існує. Під час кроку навчання в
памʼяті лежать чотири речі: **самі ваги** (потрібні всі), **градієнти**,
і **два моменти** оптимізатора Adam — а ці три потрібні лише тим вагам, які
вчаться. По 4 байти на число.

In [ ]:
rows = []
for label, kind in (('тільки голова', 'probe'), ('LoRA r=4', 'lora'), ('усі ваги', 'full')):
    m = build_arm(kind, 0)
    all_weights = sum(p.numel() for p in m.parameters())
    trained = trainable_weights(m)
    rows.append((label, all_weights, trained,
                 4 * all_weights,          # ваги
                 4 * trained,              # градієнти
                 8 * trained,              # два моменти Adam
                 4 * all_weights + 12 * trained))

print(f'{"спосіб":15}{"усього ваг":>12}{"навчуваних":>12}{"частка":>10}'
      f'{"ваги, КБ":>11}{"град+Adam, КБ":>15}{"разом, КБ":>12}')
full_trained = rows[-1][2]
for label, allw, tr, w, g, adam, total in rows:
    print(f'{label:15}{allw:12}{tr:12}{100*tr/full_trained:9.4f}%'
          f'{w/1024:11.1f}{(g+adam)/1024:15.1f}{total/1024:12.1f}')

print()
print(f'на одну вагу: повне донавчання {rows[-1][6]/rows[-1][1]:.2f} байта, '
      f'LoRA {rows[1][6]/rows[1][1]:.2f} байта')

Стовпчик «ваги, КБ» однаковий в усіх трьох рядках — заморожену вагу все одно
треба тримати, щоб порахувати вихід. А от «град+Adam» падає майже до нуля, і саме
через це повне донавчання великої моделі не влазить у карту, а LoRA влазить.

## 7 · Донавчання: однаковий бюджет усім трьом

Тепер найважливіше правило чесного порівняння. Ми даємо кожній гілці:

- **однакову кількість кроків** і однаковий розмір пачки;
- **свою сітку швидкості навчання**, і найкраще значення обираємо на
  **відкладеній** частині — тій, яку ми відклали саме для цього.

Друге правило не примха. Оптимальний крок у LoRA зазвичай на порядок більший,
ніж у повного донавчання: там навчаються нові матриці з нуля, тут — обережно
підправляються вже добрі ваги. Дати обом однаковий крок означало б навмисне
покалічити одну з них — і виміряти не метод, а власну недбалість.

Класи в нас нерівні (збоїв близько 19 %), тому втраті ставимо вагу, обернену до
частоти класу, а міряємо **macro-F1** — середнє F1 по обох класах. Звичайна
точність тут нічого б не показала: модель, що каже «не збій» завжди, дістала б
понад 80 %.

In [ ]:
FINETUNE_STEPS = 300                        # однаковий бюджет усім гілкам
LR_GRID = [1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1]   # одна сітка всім, вибір — свій кожній

# вага класу, обернена до його частоти: інакше модель просто ігнорує рідкий клас
n_fail = sum(train_y)
class_weight = torch.tensor([1.0, (len(train_y) - n_fail) / n_fail], dtype=torch.float)
print(f'вага класу «збій»: {class_weight[1]:.2f}')

def finetune(model, steps, lr, seed):
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    loss_fn = nn.CrossEntropyLoss(weight=class_weight)
    sampler = random.Random(seed)
    order = list(range(len(train_x)))
    t0 = time.process_time()
    for _ in range(steps):
        chosen = sampler.sample(order, BATCH)
        x = pad_batch([train_x[j] for j in chosen])
        y = torch.tensor([train_y[j] for j in chosen])
        loss = loss_fn(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return time.process_time() - t0

@torch.no_grad()
def macro_f1(model, xs, ys, batch=256):
    model.eval()
    predicted = []
    for i in range(0, len(xs), batch):
        predicted += model(pad_batch(xs[i:i + batch])).argmax(-1).tolist()
    model.train()
    return f1_score(ys, predicted, average='macro')

print('готово')

In [ ]:
ARMS = [('тільки голова', 'probe'), ('LoRA r=4', 'lora'), ('усі ваги', 'full')]
chosen_lr = {}

for label, kind in ARMS:
    scores = []
    for lr in LR_GRID:
        model = build_arm(kind, 0)
        finetune(model, FINETUNE_STEPS, lr, 0)
        scores.append(macro_f1(model, hold_x, hold_y))
    best = LR_GRID[max(range(len(LR_GRID)), key=lambda j: scores[j])]
    chosen_lr[label] = best
    edge = 'НА КРАЮ сітки' if best in (LR_GRID[0], LR_GRID[-1]) else 'усередині сітки'
    print(f'{label:15} ' + '  '.join(f'{lr:g}: {s:.4f}' for lr, s in zip(LR_GRID, scores))
          + f'   -> {best:g} ({edge})', flush=True)

### Дивись на всю сітку, а не лише на її найкраще число

Два зауваження до того, що надрукувалось вище, і обидва важливі.

⚠️ **Якщо найкраще значення опинилось на краю сітки** — справжній оптимум лежить
за її межами, і число цієї гілки занижене. Сітку тоді треба продовжити в той бік
і подивитись ще раз.

⚠️ **Крива по кроку не зобовʼязана бути одномодальною.** «Найкраще значення
всередині сітки» гарантує лише те, що **дві сусідні точки гірші** — і нічого
більше. Якщо між двома точками сітки є провал, жоден показник про нього не
скаже: ти просто побачиш акуратне число й підеш далі. Тому дивись на **весь**
рядок чисел, надрукований вище, а не тільки на той, що після стрілки. Ми
навмисно друкуємо всю сітку саме для цього.

In [ ]:
SEEDS = 3                     # у зошиті три зерна заради часу; на головне число треба пʼять
results = []

for label, kind in ARMS:
    scores, seconds = [], []
    weights = None
    for seed in range(SEEDS):
        model = build_arm(kind, seed)
        if weights is None:
            weights = trainable_weights(model)
        seconds.append(finetune(model, FINETUNE_STEPS, chosen_lr[label], seed))
        scores.append(macro_f1(model, test_x, test_y))
    results.append(dict(label=label, kind=kind, weights=weights,
                        lo=min(scores), hi=max(scores),
                        mean=sum(scores) / len(scores), sec=sum(seconds) / len(seconds)))
    print(f'{label:15} ваг {weights:8} · крок {chosen_lr[label]:g} · '
          f'F1 {min(scores):.4f}..{max(scores):.4f} · {seconds[0]:.0f} с процесорних',
          flush=True)

full_weights = results[-1]['weights']
print()
for r in results:
    print(f'{r["label"]:15} {100*r["weights"]/full_weights:8.4f} % ваг повного донавчання')

### Як це читати

Три числа треба дивитись **разом**, а не поодинці:

- **тільки голова** — рубіж знизу. Стільки дає передтренована модель, у якої не
  змінили жодної ваги тіла;
- **усі ваги** — рубіж зверху за цього бюджету;
- **LoRA** — де вона стала між ними, і якою ціною.

Одне число без двох інших не означає нічого. «LoRA дала 0.8» — це багато чи
мало? Відповідь залежить від того, скільки давала голова сама по собі.

⚠️ **Три зерна — це мало.** Купа з трьох прогонів систематично вужча за
справжній розкид. Якщо різниця між гілками менша за розкид усередині гілки,
різниці немає. У домашньому завданні цього робити не можна — там треба
щонайменше три зерна, а на головне твердження й усі пʼять.

## 8 · Перевірка друга: злиття точне

Найкрасивіша властивість LoRA — після навчання її можна **вдрукувати** в саму
матрицю. Поки `A` і `B` вчились, вихід шару складався з двох доданків. Але
щойно навчання скінчилось, добуток `B·A` — фіксована матриця тих самих
розмірів, що й `W`. Її просто додають, і гілки більше немає.

Це означає нуль затримки на виведенні: модель після злиття не відрізнити від
донавченої звичайним способом. Перевіримо, що це справді **тотожність**, а не
наближення: візьмемо навчену LoRA-модель, порахуємо її передбачення, потім
зіллємо ваги й порахуємо ще раз.

In [ ]:
# беремо навчену LoRA-гілку
lora_model = build_arm('lora', 0)
finetune(lora_model, FINETUNE_STEPS, chosen_lr['LoRA r=4'], 0)

sample = pad_batch(test_x[:256])
lora_model.eval()
with torch.no_grad():
    before = lora_model(sample).clone()

# зливаємо: замінюємо LoRA-обгортки на звичайні шари з матрицею W + B·A·alpha/r
merged = build_arm('probe', 0)
merged.head.load_state_dict(lora_model.head.state_dict())
with torch.no_grad():
    for name in ('Wq', 'Wv'):
        wrapper = getattr(lora_model.body.block, name)
        plain_layer = getattr(merged.body.block, name)
        plain_layer.weight.copy_(wrapper.merged_weight())
        plain_layer.bias.copy_(wrapper.base.bias)

merged.eval()
with torch.no_grad():
    after = merged(sample)

gap = (before - after).abs().max().item()
print(f'найбільша різниця виходів до й після злиття: {gap:.3e}')
assert gap < 1e-5, 'злиття мусить бути тотожним!'
print('✅ збігається: злита модель робить рівно те саме')
print()
print('ваг у злитій моделі:', sum(p.numel() for p in merged.parameters()))
print('ваг у незлитій:     ', sum(p.numel() for p in lora_model.parameters()))
lora_model.train(); merged.train()

Зверни увагу на два останні числа: у злитої моделі ваг **менше**, ніж у
незлитої, — рівно на розмір `A` і `B`. Поправка нікуди не поділася, вона просто
розчинилась у матриці.

## 9 · Малюнок: якість проти кількості навчуваних ваг

Одна картинка, у якої по горизонталі — скільки ваг ми дозволили змінювати
(логарифмічна вісь, бо числа різняться в тисячі разів), а по вертикалі —
macro-F1. Дві горизонтальні лінії — рубежі знизу й зверху.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.6))

colors = {'probe': '#c2620f', 'lora': '#c2185b', 'full': '#0f766e'}
for r in results:
    ax.errorbar(r['weights'], r['mean'],
                yerr=[[r['mean'] - r['lo']], [r['hi'] - r['mean']]],
                fmt='o', ms=8, capsize=5, color=colors[r['kind']], label=r['label'])
    ax.annotate(f"{r['mean']:.4f}", (r['weights'], r['hi']),
                textcoords='offset points', xytext=(0, 9), ha='center', fontsize=9)

ax.axhline(results[0]['mean'], ls='--', lw=1, color=colors['probe'])
ax.axhline(results[-1]['mean'], ls='--', lw=1, color=colors['full'])
ax.set_xscale('log')
ax.set_xlabel('навчуваних ваг (логарифмічна вісь)')
ax.set_ylabel('macro-F1 на перевірній частині')
ax.set_title('Скільки ваг треба чіпати')
ax.grid(alpha=.25)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

print('вуса — розкид по', SEEDS, 'зернах; точка — середнє')

## 10 · Що ми з цього дізнались

Порахуймо підсумок словами й числами.

In [ ]:
probe_r, lora_r, full_r = results
space = full_r['mean'] - probe_r['mean']          # увесь простір, який є що вигравати
gained = lora_r['mean'] - probe_r['mean']

print(f'рубіж знизу (тільки голова):  {probe_r["mean"]:.4f}')
print(f'рубіж зверху (усі ваги):      {full_r["mean"]:.4f}')
print(f'LoRA r=4:                     {lora_r["mean"]:.4f}')
print()
if space > 1e-6:
    print(f'увесь доступний простір: {space:.4f}')
    print(f'LoRA закрила з нього:    {gained:.4f}  ({100*gained/space:.1f} %)')
print(f'ціною {lora_r["weights"]} ваг замість {full_r["weights"]} — '
      f'{100*lora_r["weights"]/full_r["weights"]:.4f} %')
print()
overlap = not (lora_r['lo'] > full_r['hi'] or full_r['lo'] > lora_r['hi'])
print('купи LoRA й повного донавчання ' + ('ПЕРЕКРИВАЮТЬСЯ — на цих даних і цьому '
      'бюджеті різниці не доведено' if overlap else 'НЕ перекриваються — різниця є'))
print(f'  LoRA {lora_r["lo"]:.4f}..{lora_r["hi"]:.4f} · '
      f'повне {full_r["lo"]:.4f}..{full_r["hi"]:.4f}')
print()
print(f'усього процесорного часу на зошит: {time.process_time()-STARTED:.0f} с')

**Що тут головне.**

1. **Головне число теми — не F1, а частка ваг.** LoRA чіпає частки відсотка від
   того, що чіпає повне донавчання, і саме ця частка визначає, скільки памʼяті
   треба на навчання й скільки місця займе результат.
2. **Число нічого не варте без двох рубежів.** Сама по собі якість LoRA не
   каже нічого: треба бачити, скільки давала заморожена модель із однією
   головою (знизу) і скільки дає повна свобода (зверху).
3. **Порівняння чесне лише за однакового зусилля.** Ми дали всім трьом гілкам
   однакову кількість кроків і кожній — власний підібраний крок навчання.
   Якби ми дали всім однаковий крок, ми б виміряли не методи, а свою лінь.
4. **Два зерна замало для вироку.** Якщо купи перетинаються, чесна відповідь —
   «різниці не доведено», а не «методи однакові».

**Чого цей зошит не показує.** У нас крихітна модель, один блок і невеликий
бюджет донавчання. Виграш LoRA в памʼяті тут вимірюється кілобайтами й на
практиці нікого не рятує — рятує він на моделях у мільярди ваг, де ті самі
відсотки перетворюються на сотні гігабайтів. Форма явища відтворюється, масштаб
наслідків — ні.

## Завдання

**🟢 Рівень 1.** Додай четверту гілку — **BitFit**: заморозь усі матриці тіла,
але дозволь вчитись усім зсувам (`bias`) і параметрам нормалізації. Скільки це
ваг? Куди вона стала між рубежами?

*Зроблено, якщо:* у таблиці й на графіку зʼявився четвертий рядок із
надрукованою кількістю навчуваних ваг і купою по двох зернах.

**🟡 Рівень 2.** Пройдись рангом: `r` = 1, 2, 4, 8, 16. Побудуй криву «якість
від рангу» й знайди, з якого рангу вона перестає рости. Не забудь підбирати
крок навчання **кожному** рангу окремо.

*Зроблено, якщо:* є таблиця з пʼятьма рядками й речення про те, де крива
насичується — і чи перекриваються купи сусідніх рангів.

**🔴 Рівень 3.** Перевір гіпотезу, на якій стоїть увесь метод. Зроби **повне**
донавчання, порахуй `ΔW = W_після − W_до` для кожної матриці тіла, розклади
кожну за сингулярними числами (`torch.linalg.svdvals`) і знайди, скільки
найбільших із них тримають 90 % суми квадратів. Це **ефективний ранг** поправки.

*Зроблено, якщо:* надрукована таблиця «матриця · повний ранг · ефективний ранг
`ΔW`», і сказано вголос, чи виправдовує заміряний ефективний ранг те `r`, яке
виграло на рівні 2. Якщо не виправдовує — це теж результат, і цікавіший.